# 5.3 Sources

Sources define how energy enters the model. This tutorial combines three scalar point-source locations with one compound scalar source, runs the survey, and compares the resulting gathers. It also shows why batching matters: multiple logical sources can be solved in batches instead of one job per shot when the solver/site supports it.

By the end, you should be able to define multiple source mechanisms, batch compatible sources, and compare geometry/mechanism effects in the resulting gathers.


## Design Notes

`add_source_group(...)` adds one source per coordinate. `kind="scalar"` is the natural acoustic pressure source used in this notebook. Other physics can use directional or tensor-like mechanisms, such as `kind="vector"` for force-like sources in elastic settings or `kind="moment"` where supported. `add_compound_source(...)` creates a single logical source from multiple weighted points, which is useful for dipoles, moment-like approximations, or source arrays.

| Source pattern | Notebook API | Interpretation |
| --- | --- | --- |
| Multiple point shots | `add_source_group(kind="scalar", coords=[...])` | One logical source per coordinate. |
| Compound source | `add_compound_source(..., weights=[...])` | One logical source made from weighted point contributions. |
| Source batching | `Acquisition(max_batch=...)` | How many logical sources may be grouped into one solve batch. |

Larger batches reduce overhead when the solver can reuse setup work, but each batch still has to fit site memory. The best batch size is therefore a site/runtime choice, not a modeling property.


## Imports


In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import frequensolve as fs

u = fs.ureg


## Acoustic Model And Batched Source Acquisition

The acoustic model is enough to demonstrate scalar sources and a compound pressure dipole. Receivers are a dense pressure line so every source can be compared on the same output grid.


In [ ]:
project = fs.Project(
    name="project",
    pretty_name="source_mechanisms",
    path="./scratch/tutorials/sources",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="source_mechanisms",
    physics="acoustic",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.0])
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(name="water", properties={"Vp": 1.5 * u.km / u.s, "Rho": 1.0 * u.g / u.cm**3})
model.add_surface(name="interface", depth=0.25 * u.km)
model.add_layer(name="basement", properties={"Vp": 2.4 * u.km / u.s, "Rho": 2.2 * u.g / u.cm**3})
model.add_surface(name="bottom", depth=0.5 * u.km)
sim += model

sim += model.hex_mesh_generator([8, 4])
sim.mesh.set_adapt(elems_per_wave=2.0, order=4, f_low=5.0, f_high=30.0)
sim.mesh.set_source_grading(d0=0.02, d1=0.08, mult=2.0)
sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(conditions=["pml"], boundaries=["x_min", "x_max", "z_max"], pml_wavelengths=0.75)

model.plot("vp", figsize=(7, 3), aspect="equal")


## Define Multiple Source Mechanisms

The first call adds three scalar shots. The compound source is a two-point weighted source; with weights `[1, -1]` it behaves like a compact dipole in this simple acoustic example. `max_batch=4` allows these four logical sources to be submitted efficiently as one batch when supported.


In [ ]:
acq = fs.Acquisition(max_batch=4)
acq.add_source_group(kind="scalar", coords=[[0.25, 0.05], [0.5, 0.05], [0.75, 0.05]])
acq.add_compound_source(
    kind="scalar",
    coords=[[0.45, 0.08], [0.55, 0.08]],
    weights=[1.0, -1.0],
)

hydrophone = fs.ReceiverNode(name="hydrophone")
hydrophone.add_component(name="p", field="pressure")
receiver_coords = [[x, 0.04] for x in np.linspace(0.1, 0.9, 81)]
acq.add_receiver_group(name="line", device=hydrophone, coords=receiver_coords)
sim += acq

{
    "logical_sources": len(acq.source_groups),
    "max_batch": acq.max_batch,
    "dense_trace_count": len(acq.source_groups) * len(receiver_coords),
}


## Run The Source Comparison

The result contains one source id per logical source. Use `traces.sources(group)` to discover the ids written by the solver instead of assuming a numbering convention in downstream analysis.

This run also demonstrates why source batching is a simulation setting rather than a plotting detail. The receiver geometry is identical for every source, but `max_batch=4` lets the site/solver evaluate compatible sources together when supported. If a site lowers the batch size for memory reasons, the trace API remains the same; only execution scheduling changes.


In [ ]:
sim += fs.Discretization()
sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

site = fs.LocalSite(shutdown_on_completion=True, verbose=True)
job = fs.TimeDomainJob(
    name="time_sources",
    simulation=sim,
    f_min=0.0,
    f_max=30.0,
    T_max=0.9,
)
result = site.submit(job).wait()
traces = result.traces(upscale=4)
traces.summary


## Plot One Gather Per Source

Comparing gathers side by side makes source position and source mechanism differences visible. The three scalar sources shift the moveout apex; the compound source changes polarity and radiation relative to a single point source.


In [ ]:
wavelet = fs.RickerWavelet(f=12.0)
group = "line"
component = "p"
source_ids = traces.sources(group)
source_gathers = {
    source_id: traces.td(group, component, source_id, wavelet, upscale=4, T_max=0.9)
    for source_id in source_ids
}

A = 2.0 * max(float(np.nanstd(np.real(gather.values))) for gather in source_gathers.values())
fig, axes = plt.subplots(1, len(source_gathers), figsize=(4.3 * len(source_gathers), 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (source_id, gather) in zip(axes, source_gathers.items()):
    fs.plot_gather(gather, ax=ax, A=A, cmap="gray", title=f"source {source_id}")
fig.tight_layout()


## Overlay Center-Receiver Responses

Line plots are useful for checking polarity and wavelet timing. This overlay uses one receiver index for every source id in the result.


In [ ]:
center = len(receiver_coords) // 2
fig, ax = plt.subplots(figsize=(9, 4))
for source_id, gather in source_gathers.items():
    channel = gather.isel(receiver=center)
    ax.plot(channel["time"].values, np.real(channel.values), label=f"source {source_id}")
ax.set_xlabel("Time")
ax.set_ylabel("Pressure")
ax.set_title(f"Center receiver response for {len(source_gathers)} source mechanisms")
ax.legend()
fig.tight_layout()


## Result Review Checklist

The source tutorial is successful when the result shows both geometry effects and mechanism effects. The side-by-side gathers should move laterally with the three point shots, while the compound source should show a different polarity/radiation pattern because it is a weighted pair, not a fourth point shot.

| Artifact | What to inspect |
| --- | --- |
| `traces.sources(group)` | Source ids are discovered from the trace file rather than assumed. |
| Gather panels | Moveout apex shifts with point-source location. |
| Center-receiver overlay | Polarity and timing differences are visible at one receiver. |
| Acquisition payload | `max_batch` records batching intent; changing batch size should not change trace semantics. |
